# dbt Jinja Processor for Redshift-to-Trino Migration

This notebook is a public-safe prototype for deterministic dbt/Jinja cleanup during a Redshift-to-Trino/Apache Iceberg migration.

The goal is not to execute dbt. The goal is to safely render enough Jinja to remove or rewrite known migration-incompatible patterns while preserving dbt macros such as `ref`, `source`, `var`, `dbt_utils`, and adapter helpers for later compilation.


## Migration Rules Captured Here

Redshift dbt projects often contain model configuration that does not apply to Trino/Iceberg. The processor can remove or rewrite these keys from `{{ config(...) }}` blocks:

- `dist`
- `distkey`
- `dist_key`
- `sort`
- `sortkey`
- `sort_key`
- `materialize`
- `strategy`

For staging or compatibility views, the processor can also remove incremental blocks such as:

```jinja
{% if is_incremental() %}
  ...
{% endif %}
```

This is useful when the migration intentionally converts a Redshift incremental model into a Trino/Iceberg view or when physical layout behavior moves from warehouse-specific SQL into the lakehouse operating model.


## Imports


In [ ]:
from __future__ import annotations

import glob
import re
import traceback
from pathlib import Path
from typing import Any, Iterable, Optional

from jinja2 import BaseLoader, Environment


## Processor

`DbtJinjaProcessor` simulates a small subset of the dbt Jinja runtime. It keeps common dbt constructs as literal Jinja strings while allowing deterministic cleanup of `config(...)` blocks and incremental conditionals.


In [ ]:
class DbtJinjaProcessor:
    """Render enough dbt/Jinja to clean migration-specific SQL patterns.

    The processor intentionally preserves dbt expressions such as `ref`, `source`,
    `var`, `dbt_utils.*`, and `dbt.*` so the output can still be compiled by dbt
    later. It only rewrites the behaviors requested through constructor options.
    """

    DEFAULT_PASSTHROUGH_MACROS = (
        "date_formatting",
        "business_minutes_between",
        "normalize_email",
        "hash_key",
        "target",
    )

    def __init__(
        self,
        excluded_config_args: Optional[Iterable[str]] = None,
        add_config_args: Optional[dict[str, Any]] = None,
        is_incremental: bool = False,
        force_config_if_missing: bool = True,
        passthrough_macro_names: Optional[Iterable[str]] = None,
    ) -> None:
        self.excluded_config_args = set(excluded_config_args or [])
        self.add_config_args = add_config_args or {}
        self.is_incremental = is_incremental
        self.force_config_if_missing = force_config_if_missing
        self.passthrough_macro_names = tuple(
            passthrough_macro_names or self.DEFAULT_PASSTHROUGH_MACROS
        )

        self.env = Environment(loader=BaseLoader())
        self._set_globals()

    def _set_globals(self) -> None:
        self.env.globals["source"] = self._mock_source
        self.env.globals["ref"] = self._mock_ref
        self.env.globals["var"] = self._mock_var
        self.env.globals["config"] = self._mock_config
        self.env.globals["is_incremental"] = lambda: self.is_incremental
        self.env.globals["dbt_utils"] = self._get_namespace("dbt_utils")
        self.env.globals["dbt"] = self._get_namespace("dbt")

        for macro_name in self.passthrough_macro_names:
            self.env.globals[macro_name] = self._make_passthrough_macro(macro_name)

    @staticmethod
    def _format_value(value: Any) -> str:
        if isinstance(value, list):
            return "[" + ", ".join(repr(item) for item in value) + "]"
        if isinstance(value, tuple):
            return "(" + ", ".join(repr(item) for item in value) + ")"
        if isinstance(value, dict):
            pairs = [f"{repr(k)}: {repr(v)}" for k, v in value.items()]
            return "{" + ", ".join(pairs) + "}"
        return repr(value)

    def _format_call(self, name: str, *args: Any, **kwargs: Any) -> str:
        positional = [self._format_value(arg) for arg in args]
        keyword = [f"{key}={self._format_value(value)}" for key, value in kwargs.items()]
        return f"{{{{ {name}({', '.join(positional + keyword)}) }}}}"

    def _make_passthrough_macro(self, name: str):
        def macro(*args: Any, **kwargs: Any) -> str:
            return self._format_call(name, *args, **kwargs)

        return macro

    def _get_namespace(self, namespace: str):
        processor = self

        class PassthroughNamespace:
            def __getattr__(self, macro_name: str):
                return lambda *args, **kwargs: processor._format_call(
                    f"{namespace}.{macro_name}", *args, **kwargs
                )

        return PassthroughNamespace()

    def _mock_source(self, source_name: str, table_name: str) -> str:
        return self._format_call("source", source_name, table_name)

    def _mock_ref(self, model_name: str) -> str:
        return self._format_call("ref", model_name)

    def _mock_var(self, var_name: str, default: Any = None) -> str:
        if default is None:
            return self._format_call("var", var_name)
        return self._format_call("var", var_name, default)

    def _mock_config(self, **kwargs: Any) -> str:
        filtered_kwargs = {
            key: value
            for key, value in kwargs.items()
            if key not in self.excluded_config_args
        }
        filtered_kwargs.update(self.add_config_args)

        if not filtered_kwargs:
            return ""

        arg_strings = [
            f"{key}={self._format_value(value)}"
            for key, value in filtered_kwargs.items()
        ]
        return f"{{{{ config({', '.join(arg_strings)}) }}}}"

    @staticmethod
    def _has_config(template_str: str) -> bool:
        return bool(re.search(r"{{\s*config\s*\(", template_str))

    @staticmethod
    def _has_incremental_block(template_str: str) -> bool:
        return bool(re.search(r"{%\s*if\s+is_incremental\(\)\s*%}", template_str))

    @staticmethod
    def _remove_incremental_blocks(template_str: str) -> str:
        return re.sub(
            r"{%\s*if\s+is_incremental\(\)\s*%}.*?{%\s*endif\s*%}",
            "",
            template_str,
            flags=re.DOTALL,
        )

    @staticmethod
    def _collapse_blank_lines(sql_text: str) -> str:
        return re.sub(r"\n{3,}", "\n\n", sql_text).strip() + "\n"

    def process_template_str(self, template_str: str) -> str:
        modifying_config = bool(self.excluded_config_args or self.add_config_args)
        should_force_config = (
            self.force_config_if_missing
            and self.add_config_args
            and not self._has_config(template_str)
        )
        should_remove_incremental = (
            not self.is_incremental and self._has_incremental_block(template_str)
        )

        if not modifying_config and not should_force_config and not should_remove_incremental:
            return template_str

        working_sql = template_str

        if should_force_config:
            config_args = [
                f"{key}={self._format_value(value)}"
                for key, value in self.add_config_args.items()
            ]
            working_sql = f"{{{{ config({', '.join(config_args)}) }}}}\n" + working_sql

        if should_remove_incremental:
            working_sql = self._remove_incremental_blocks(working_sql)

        rendered = self.env.from_string(working_sql).render()
        return self._collapse_blank_lines(rendered)


## Example Model Before Translation

This example is intentionally generic. It demonstrates common Redshift-oriented dbt patterns: incremental config, physical layout keys, dbt utility macros, and an `is_incremental()` block.


In [ ]:
example_model = """
{{ config(
    materialized='incremental',
    unique_key='order_id',
    strategy='merge',
    dist='customer_id',
    sort=['updated_at'],
    tags=['lakehouse_migration']
) }}

SELECT
    order_id,
    customer_id,
    {{ dbt_utils.generate_surrogate_key(['customer_id', 'order_id']) }} AS order_key,
    {{ normalize_email('customer_email') }} AS customer_email_normalized,
    CAST(updated_at AS {{ dbt.type_timestamp() }}) AS updated_at_ts,
    order_total
FROM {{ source('raw_app', 'orders') }}
WHERE order_status IS NOT NULL

{% if is_incremental() %}
  AND updated_at >= DATE_ADD('day', -2, CURRENT_DATE)
{% endif %}
"""


## Convert a Model to a Trino/Iceberg View

This mode removes Redshift-only physical layout config and strips incremental logic because the target object is a view.


In [ ]:
view_processor = DbtJinjaProcessor(
    excluded_config_args=[
        "materialized",
        "strategy",
        "dist",
        "distkey",
        "dist_key",
        "sort",
        "sortkey",
        "sort_key",
        "on_schema_change",
    ],
    add_config_args={"materialized": "view"},
    is_incremental=False,
)

translated_view_model = view_processor.process_template_str(example_model)
print(translated_view_model)


## Keep Incremental Logic but Remove Redshift-Only Config

This mode keeps the `is_incremental()` block while removing Redshift-only physical layout config. It is useful when the Trino/Iceberg target still uses an incremental materialization.


In [ ]:
incremental_processor = DbtJinjaProcessor(
    excluded_config_args=[
        "dist",
        "distkey",
        "dist_key",
        "sort",
        "sortkey",
        "sort_key",
    ],
    is_incremental=True,
)

translated_incremental_model = incremental_processor.process_template_str(example_model)
print(translated_incremental_model)


## Batch Processing Helper

The helper below is safe by default: `dry_run=True` means it reports what would change without writing files. Set `dry_run=False` only after reviewing the output, preferably against a migration branch.


In [ ]:
def process_sql_files(
    directory: str | Path,
    processor: DbtJinjaProcessor,
    error_log_path: str | Path = "error_log_jinja.txt",
    dry_run: bool = True,
    output_directory: str | Path | None = None,
) -> dict[str, list[str]]:
    """Process dbt SQL files recursively.

    Args:
        directory: Root folder containing dbt model SQL files.
        processor: Configured `DbtJinjaProcessor` instance.
        error_log_path: File where processing errors are written.
        dry_run: If true, do not write changed SQL back to disk.
        output_directory: Optional folder for processed copies. If omitted and
            `dry_run=False`, files are overwritten in place.
    """
    root = Path(directory)
    sql_files = [Path(path) for path in glob.glob(str(root / "**" / "*.sql"), recursive=True)]
    output_root = Path(output_directory) if output_directory else None
    error_log = Path(error_log_path)

    processed_files: list[str] = []
    changed_files: list[str] = []
    failed_files: list[str] = []

    for sql_file in sql_files:
        try:
            original_content = sql_file.read_text(encoding="utf-8")
            processed_content = processor.process_template_str(original_content)
            processed_files.append(str(sql_file))

            if processed_content != original_content:
                changed_files.append(str(sql_file))

                if not dry_run:
                    if output_root:
                        target_file = output_root / sql_file.relative_to(root)
                        target_file.parent.mkdir(parents=True, exist_ok=True)
                    else:
                        target_file = sql_file
                    target_file.write_text(processed_content, encoding="utf-8")

        except Exception as exc:
            failed_files.append(str(sql_file))
            error_message = (
                f"Failed to process {sql_file}: {exc}\n{traceback.format_exc()}\n"
            )
            with error_log.open("a", encoding="utf-8") as error_file:
                error_file.write(error_message)

    return {
        "processed": processed_files,
        "changed": changed_files,
        "failed": failed_files,
    }


# Example usage:
# processor = DbtJinjaProcessor(
#     excluded_config_args=["dist", "sort", "distkey", "sortkey"],
#     is_incremental=True,
# )
# summary = process_sql_files("models", processor, dry_run=True)
# summary


## Public Portfolio Notes

This notebook is intentionally a prototype, not a package. The production version of this idea would likely become a small script or YAML-driven rule engine with tests around each translation rule.

Recommended next extraction:

```text
dbt-translation-engine/
  dbt_jinja_processor.py
  redshift_to_trino_rules.yml
  fixtures/
    before.sql
    after.sql
```
